In [6]:
import numpy as np
import glob
import os

import 
import glob
import numpy as np
import matplotlib.pyplot as plt # Gakktu úr skugga um að þetta sé líka til staðar
def extract_and_visualize_sentinel2(zip_path, output_dir=None):
    """
    Extract a Sentinel-2 ZIP file and create an RGB visualization.
    
    Parameters:
    -----------
    zip_path : str
        Path to the Sentinel-2 ZIP file
    output_dir : str, optional
        Directory to extract to. If None, extracts to same directory as ZIP.
    
    Returns:
    --------
    str : Path to extracted SAFE directory
    """
    import zipfile
    import rasterio
    from rasterio.plot import show
    
    if output_dir is None:
        output_dir = os.path.dirname(zip_path)
    
    # Extract ZIP file
    print(f"Extracting: {os.path.basename(zip_path)}")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Get the SAFE directory name (first item in the archive)
        safe_dir = zip_ref.namelist()[0].split('/')[0]
        safe_path = os.path.join(output_dir, safe_dir)
        
        if os.path.exists(safe_path):
            print(f"⚠ Already extracted: {safe_dir}")
        else:
            zip_ref.extractall(output_dir)
            print(f"✓ Extracted to: {safe_path}")
    
    return safe_path


def visualize_sentinel2_rgb(safe_path, figsize=(12, 12)):
    """
    Create an RGB visualization from Sentinel-2 SAFE directory.
    
    Uses bands B04 (Red), B03 (Green), B02 (Blue) at 10m resolution.
    
    Parameters:
    -----------
    safe_path : str
        Path to the extracted .SAFE directory
    figsize : tuple
        Figure size for the plot
    """
    import rasterio
    from rasterio.enums import Resampling
    
    # Find the 10m resolution bands (B02, B03, B04)
    # Path pattern: .SAFE/GRANULE/*/IMG_DATA/R10m/*_B0X_10m.jp2
    granule_path = os.path.join(safe_path, 'GRANULE')
    
    if not os.path.exists(granule_path):
        print(f"❌ GRANULE directory not found in {safe_path}")
        return
    
    # Get the tile subdirectory
    tile_dirs = [d for d in os.listdir(granule_path) if os.path.isdir(os.path.join(granule_path, d))]
    if not tile_dirs:
        print("❌ No tile directories found")
        return
    
    tile_dir = os.path.join(granule_path, tile_dirs[0])
    
    # Try R10m directory first (newer format), then IMG_DATA (older format)
    r10m_path = os.path.join(tile_dir, 'IMG_DATA', 'R10m')
    img_data_path = os.path.join(tile_dir, 'IMG_DATA')
    
    if os.path.exists(r10m_path):
        band_dir = r10m_path
        band_pattern = '*_B0{}_10m.jp2'
    else:
        band_dir = img_data_path
        band_pattern = '*_B0{}.jp2'
    
    # Find band files
    bands = {}
    for band_num in ['2', '3', '4']:
        pattern = os.path.join(band_dir, band_pattern.format(band_num))
        matches = glob.glob(pattern)
        if matches:
            bands[f'B0{band_num}'] = matches[0]
        else:
            # Try alternative pattern for older format
            alt_pattern = os.path.join(band_dir, f'*B0{band_num}*.jp2')
            alt_matches = glob.glob(alt_pattern)
            if alt_matches:
                bands[f'B0{band_num}'] = alt_matches[0]
    
    if len(bands) < 3:
        print(f"❌ Could not find all RGB bands. Found: {list(bands.keys())}")
        print(f"   Searched in: {band_dir}")
        return
    
    print(f"✓ Found bands: {list(bands.keys())}")
    
    # Read bands with downsampling for visualization (full resolution can be huge)
    downsample_factor = 10  # Read at 1/10 resolution for faster display
    
    rgb_bands = []
    for band_name in ['B04', 'B03', 'B02']:  # RGB order
        with rasterio.open(bands[band_name]) as src:
            # Calculate new dimensions
            new_height = src.height // downsample_factor
            new_width = src.width // downsample_factor
            
            # Read with resampling
            band_data = src.read(
                1,
                out_shape=(new_height, new_width),
                resampling=Resampling.average
            )
            rgb_bands.append(band_data)
    
    # Stack into RGB array
    rgb = np.stack(rgb_bands, axis=-1)
    
    # Normalize for visualization (typical Sentinel-2 values range 0-10000)
    # Use percentile-based stretching for better visualization
    p2, p98 = np.percentile(rgb[rgb > 0], (2, 98))
    rgb_normalized = np.clip((rgb - p2) / (p98 - p2), 0, 1)
    
    # Create visualization
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    
    # RGB composite
    axes[0].imshow(rgb_normalized)
    axes[0].set_title('Sentinel-2 RGB Composite (B4-B3-B2)', fontsize=12)
    axes[0].axis('off')
    
    # False color (NIR-Red-Green) if available
    # For now, show a histogram of the data
    axes[1].hist(rgb[:,:,0].flatten()[::100], bins=50, alpha=0.7, label='Red (B04)', color='red')
    axes[1].hist(rgb[:,:,1].flatten()[::100], bins=50, alpha=0.7, label='Green (B03)', color='green')
    axes[1].hist(rgb[:,:,2].flatten()[::100], bins=50, alpha=0.7, label='Blue (B02)', color='blue')
    axes[1].set_xlabel('Reflectance Value')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Band Value Distribution', fontsize=12)
    axes[1].legend()
    axes[1].set_xlim(0, 5000)
    
    plt.tight_layout()
    plt.savefig(os.path.join(os.environ['PROJECT_training2600'],  f"{os.environ['USER']}/results/S2_vis.png"))
    
    # Print some metadata
    with rasterio.open(bands['B04']) as src:
        print(f"\nImage Metadata:")
        print(f"  Original size: {src.width} x {src.height} pixels")
        print(f"  Resolution: {src.res[0]}m x {src.res[1]}m")
        print(f"  CRS: {src.crs}")
        print(f"  Bounds: {src.bounds}")
    
    return rgb_normalized

print("✓ Visualization functions defined")

✓ Visualization functions defined


In [11]:
# 1. Skilgreindu slóðina á ZIP skrána þína handvirkt
# Breyttu þessu í rétta nafnið á skránni sem er inni í data möppunni
manual_zip_name = "S2B_MSIL2A_20181012T094029_N0500_R036_T33TYN_20230730T202118.SAFE.zip" 
download_dir = "/p/project1/training2600/TeamGudnason/data"
zip_path = os.path.join(download_dir, manual_zip_name)

# 2. Keyrðu myndgreininguna beint
if os.path.exists(zip_path):
    if is_valid_zipfile(zip_path):
        print(f"✓ Fann ZIP skrá: {os.path.basename(zip_path)}")
        print("=" * 60)
        
        # Nota fallið úr kóðanum þínum til að afþjappa
        safe_path = extract_and_visualize_sentinel2(zip_path)
        
        if safe_path and os.path.isdir(safe_path):
            print(f"\nVisualizing: {os.path.basename(safe_path)}")
            print("\nCreating RGB visualization...")
            
            # Nota fallið úr kóðanum þínum til að teikna upp
            visualize_sentinel2_rgb(safe_path)
            
            print("\n" + "=" * 60)
            print("✓ Visualization complete!")
        else:
            print("❌ Gat ekki afþjappað .SAFE möppunni.")
    else:
        print(f"❌ Skráin er ekki gild ZIP skrá: {zip_path}")
else:
    print(f"❌ ZIP skráin fannst ekki á þessari slóð: {zip_path}")

✓ Fann ZIP skrá: S2B_MSIL2A_20181012T094029_N0500_R036_T33TYN_20230730T202118.SAFE.zip
Extracting: S2B_MSIL2A_20181012T094029_N0500_R036_T33TYN_20230730T202118.SAFE.zip
✓ Extracted to: /p/project1/training2600/TeamGudnason/data/S2B_MSIL2A_20181012T094029_N0500_R036_T33TYN_20230730T202118.SAFE

Visualizing: S2B_MSIL2A_20181012T094029_N0500_R036_T33TYN_20230730T202118.SAFE

Creating RGB visualization...
✓ Found bands: ['B02', 'B03', 'B04']

Image Metadata:
  Original size: 10980 x 10980 pixels
  Resolution: 10.0m x 10.0m
  CRS: EPSG:32633
  Bounds: BoundingBox(left=699960.0, bottom=5190240.0, right=809760.0, top=5300040.0)

✓ Visualization complete!


In [1]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Load eitt patch
from pathlib import Path
import numpy as np

FINAL_DIR = Path("/p/scratch/training2600/TeamGudnason/training_data/final_block_64")
patches = np.load(FINAL_DIR / "patches_train.npz")['patches']
labels  = np.load(FINAL_DIR / "labels_train.npy")

# Taka eitt patch af hverjum klasa
CLASS_NAMES = {2: 'Urban', 12: 'Arable land', 18: 'Pastures', 23: 'Forest'}
TARGET_CLASSES = [2, 12, 18, 23]

fig, axes = plt.subplots(3, 4, figsize=(14, 10))

for col, cls in enumerate(TARGET_CLASSES):
    idx = np.where(labels == cls)[0][0]
    patch = patches[idx]  # (64, 64, 16)

    # April (bands 0-3: B02,B03,B04,B08)
    rgb_apr = patch[:, :, [2, 1, 0]]  # B04, B03, B02
    rgb_apr = (rgb_apr - rgb_apr.min()) / (rgb_apr.max() - rgb_apr.min() + 1e-6)

    # August (bands 8-11)
    rgb_aug = patch[:, :, [10, 9, 8]]
    rgb_aug = (rgb_aug - rgb_aug.min()) / (rgb_aug.max() - rgb_aug.min() + 1e-6)

    # October (bands 12-15)
    rgb_oct = patch[:, :, [14, 13, 12]]
    rgb_oct = (rgb_oct - rgb_oct.min()) / (rgb_oct.max() - rgb_oct.min() + 1e-6)

    axes[0, col].imshow(rgb_apr)
    axes[0, col].set_title(CLASS_NAMES[cls], fontsize=11, fontweight='bold')
    axes[0, col].set_ylabel('April' if col == 0 else '', fontsize=10)
    axes[0, col].axis('off')

    axes[1, col].imshow(rgb_aug)
    axes[1, col].set_ylabel('August' if col == 0 else '', fontsize=10)
    axes[1, col].axis('off')

    axes[2, col].imshow(rgb_oct)
    axes[2, col].set_ylabel('October' if col == 0 else '', fontsize=10)
    axes[2, col].axis('off')

plt.suptitle('Sample 64×64 patches (640×640 m) across seasons and land cover classes\nSentinel-2 RGB composite (B04/B03/B02)', fontsize=12)
plt.tight_layout()
plt.savefig('/p/scratch/training2600/TeamGudnason/results/sample_patches_64x64.png',
            dpi=150, bbox_inches='tight')
print("Vistað.")

Vistað.


In [3]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import rasterio
from pathlib import Path

ALIGNED_DIR = Path("/p/scratch/training2600/TeamGudnason/data/aligned_data")
tile = "S2B_MSIL2A_20180408T095029_N0500_R079_T33TYN_20230906T162829"

s2_path     = ALIGNED_DIR / f"{tile}_stacked.tif"
corine_path = ALIGNED_DIR / f"corine_aligned_{tile}.tif"

DOWNSAMPLE = 20  # 10980 / 20 = 549 pixels -- handanlegt
BLOCK_SIZE_PX = 640 // 10  # 640m / 10m = 64 pixels á full res --> 64/20 = ~3 pixels á downsampled

with rasterio.open(s2_path) as src:
    h = src.height // DOWNSAMPLE
    w = src.width  // DOWNSAMPLE
    from rasterio.enums import Resampling
    s2 = src.read([3, 2, 1], out_shape=(3, h, w), resampling=Resampling.average)

with rasterio.open(corine_path) as src:
    corine = src.read(1, out_shape=(h, w), resampling=Resampling.nearest)

# Normalize RGB
s2 = s2.transpose(1, 2, 0).astype(np.float32)
p2, p98 = np.percentile(s2[s2 > 0], (2, 98))
s2 = np.clip((s2 - p2) / (p98 - p2), 0, 1)

# CORINE color map
CORINE_COLORS = {
    2: [0.9, 0.1, 0.1], 3: [1.0, 0.0, 0.0],
    12: [1.0, 1.0, 0.6], 18: [0.8, 1.0, 0.6],
    23: [0.0, 0.6, 0.0], 40: [0.4, 0.8, 1.0],
    41: [0.0, 0.4, 1.0],
}
corine_rgb = np.zeros((h, w, 3), dtype=np.float32)
for code, color in CORINE_COLORS.items():
    mask = corine == code
    corine_rgb[mask] = color
corine_rgb[corine == 0] = [0.9, 0.9, 0.9]

# Block grid overlay
block_px = 64 // DOWNSAMPLE  # block size í downsampled pixels
block_overlay = s2.copy()
for y in range(0, h, block_px):
    block_overlay[y:y+1, :, :] = [0.3, 0.3, 0.8]
for x in range(0, w, block_px):
    block_overlay[:, x:x+1, :] = [0.3, 0.3, 0.8]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(s2)
axes[0].set_title('Sentinel-2 RGB (B04, B03, B02)', fontsize=11)
axes[0].axis('off')

axes[1].imshow(corine_rgb)
axes[1].set_title('Aligned CORINE Land Cover', fontsize=11)
axes[1].axis('off')

axes[2].imshow(block_overlay)
axes[2].set_title('S2 with 64×64 block grid\n(each block = 640×640 m)', fontsize=11)
axes[2].axis('off')

plt.suptitle(f'Alignment: {tile[:40]}...', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('/p/scratch/training2600/TeamGudnason/results/alignment_64x64_blocks.png',
            dpi=150, bbox_inches='tight')
print("Vistað.")

Vistað.
